## Descarga de datos de la API de Google Maps

In [5]:
import googlemaps
import os
from datetime import datetime
import pandas as pd
from dotenv import load_dotenv

# 1. Configuración Inicial
load_dotenv()
api_key = os.getenv('GOOGLE_MAPS_API_KEY')
gmaps = googlemaps.Client(key=api_key)

In [ ]:
import googlemaps
import os
from datetime import datetime
import pandas as pd
from dotenv import load_dotenv

# 1. Configuración Inicial
load_dotenv()
api_key = os.getenv('GOOGLE_MAPS_API_KEY')
gmaps = googlemaps.Client(key=api_key)

# --- CONFIGURACIÓN DE PUNTOS CLAVE ---
origen_query = "Terminal de Ómnibus Mariano Moreno, Rosario"
# Radio central para la búsqueda (Echesortu)
radio_query = "MO Pet Shopping - Sucursal Echesortu"

# Geocodificamos el origen para obtener su Place ID y Coordenadas
res_origen_geo = gmaps.geocode(origen_query)
origen_id = res_origen_geo[0]['place_id']
origen_coords = res_origen_geo[0]['geometry']['location']

# Geocodificamos el centro del radio de búsqueda
res_radio = gmaps.geocode(radio_query)
location_busqueda = res_radio[0]['geometry']['location']

# 2. Definición de Categorías
categorias_busqueda = [
    {'label': 'Restaurante', 'type': 'restaurant', 'keyword': 'parrilla cena almuerzo gourmet pizza'},
    {'label': 'Museo', 'type': 'museum', 'keyword': 'museo'},
    {'label': 'Heladería', 'type': None, 'keyword': 'heladeria artesanal'},
    {'label': 'Cervecería', 'type': 'bar', 'keyword': 'cerveceria artesanal comida'}
]

datos_lugares = []
place_ids_vistos = set()

# --- FUNCIÓN AUXILIAR PARA PROCESAR DETALLES ---
def obtener_info_lugar(p_id, etiqueta):
    try:
        detalles = gmaps.place(
            place_id=p_id, 
            fields=['name', 'geometry', 'formatted_address', 'rating', 'opening_hours'],
            language='es'
        ).get('result', {})

        dias_semana = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
        horarios_semanales = {dia: [] for dia in dias_semana}
        periodos = detalles.get('opening_hours', {}).get('periods', [])

        if periodos:
            for p in periodos:
                idx_google = p['open']['day']
                idx_nuestro = (idx_google - 1) % 7 
                nombre_dia = dias_semana[idx_nuestro]
                h_abre = p['open']['time']
                h_cierra = p.get('close', {}).get('time', '24h')
                horarios_semanales[nombre_dia].append({
                    'apertura': f"{h_abre[:2]}:{h_abre[2:]}",
                    'cierre': f"{h_cierra[:2]}:{h_cierra[2:]}" if h_cierra != '24h' else '24h'
                })
            for dia in dias_semana:
                if not horarios_semanales[dia]: horarios_semanales[dia] = "Cerrado"
        else:
            horarios_semanales = "No disponible"

        return {
            'nombre': detalles.get('name'),
            'tipo': etiqueta,
            'coords': detalles.get('geometry', {}).get('location'),
            'direccion': detalles.get('formatted_address'),
            'puntaje': detalles.get('rating', 0),
            'horarios_semana': horarios_semanales
        }
    except:
        return None

# --- PASO 3: PROCESAR EL ORIGEN PRIMERO ---
print("Procesando punto de origen...")
info_origen = obtener_info_lugar(origen_id, "Punto de Partida")
if info_origen:
    datos_lugares.append(info_origen)
    place_ids_vistos.add(origen_id)

# --- PASO 4: BUCLE DE BÚSQUEDA ---
print(f"Buscando lugares de interés en Rosario...")
for cat in categorias_busqueda:
    busqueda = gmaps.places_nearby(
        location=location_busqueda,
        radius=3000,
        type=cat['type'],
        keyword=cat['keyword'],
        language='es'
    )

    for item in busqueda.get('results', []):
        pid = item['place_id']
        if pid in place_ids_vistos: continue
        
        # Filtro de restaurante (evitar comida al paso)
        tipos = item.get('types', [])
        if cat['label'] == 'Restaurante' and ('meal_takeaway' in tipos and 'restaurant' not in tipos):
            continue

        lugar_procesado = obtener_info_lugar(pid, cat['label'])
        if lugar_procesado:
            datos_lugares.append(lugar_procesado)
            place_ids_vistos.add(pid)

# 5. Visualización
df = pd.DataFrame(datos_lugares)
# El origen siempre quedará en la primera fila (índice 0)
print(f"\nSe encontraron {len(df) - 1} lugares + el punto de origen.")

print(df[['nombre', 'tipo', 'puntaje']].head(15))

Procesando punto de origen...
Buscando lugares de interés en Rosario...

Se encontraron 75 lugares + el punto de origen.
                               nombre              tipo  puntaje
0                    Rosario Santa Fe  Punto de Partida      3.9
1             Parrilla "La Estancia".       Restaurante      4.3
2       Chicharra Asador A Las Brasas       Restaurante      4.4
3    Parrilla Alvear y Nueve de Julio       Restaurante      4.3
4      Parrilla Restaurant La Zorrita       Restaurante      4.8
5                        Los Jardines       Restaurante      4.3
6                  Parrilla Don Ferro       Restaurante      4.4
7                        Picado Fino.       Restaurante      4.4
8             Restaurante Doppio Zero       Restaurante      4.3
9         Pizzeria Inmortales Rosario       Restaurante      4.8
10  La Huella - Tradición en Parrilla       Restaurante      4.2
11         La Gran Argentina Pizzería       Restaurante      4.2
12         La Vecchia Cucina Pizze

In [ ]:
# Se guardan el dataframe completo en un archivo .txt para análisis posterior
df.to_csv('data/df_completo.txt', index=False, sep='\t')


In [ ]:
import pandas as pd
import ast

# Carga del dataframe para análisis posterior
df = pd.read_csv('data/df_completo.txt', sep='\t')

# Convertimos los strings de nuevo a diccionarios
# Usamos literal_eval porque es más seguro que eval()
def safe_eval(val):
    try:
        if pd.isna(val) or val == "No disponible":
            return val
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return val

df['coords'] = df['coords'].apply(safe_eval)
df['horarios_semana'] = df['horarios_semana'].apply(safe_eval)

# Ahora ya puedes correr el código de transformación que te pasé antes
print(type(df.iloc[0]['coords'])) # Debería decir <class 'dict'>

<class 'dict'>


## Filtrado de los puntos de interés

In [ ]:
# 1. Separamos el Punto de Origen (para que no entre en la competencia del Top 5)
origen = df[df['tipo'] == 'Punto de Partida']

# 2. Filtramos los lugares que NO son el origen
lugares_interes = df[df['tipo'] != 'Punto de Partida']

# 3. Agrupamos por 'tipo' y tomamos los 5 con mejor puntaje de cada categoría
# Usamos sort_values para asegurar que los más altos queden arriba antes del head(5)
top_por_categoria = lugares_interes.sort_values(['tipo', 'puntaje'], ascending=[True, False])
top_5_lugares = top_por_categoria.groupby('tipo').head(5)

# 4. Concatenamos el origen con nuestra nueva selección
df_final = pd.concat([origen, top_5_lugares]).reset_index(drop=True)


print(f"Lista final: {len(df_final)} nodos (Origen + 5 de cada categoría).")
print("\nResumen de la selección:")
print(df_final[['tipo', 'nombre', 'puntaje']])

Lista final: 21 nodos (Origen + 5 de cada categoría).

Resumen de la selección:
                tipo                                             nombre  \
0   Punto de Partida                                   Rosario Santa Fe   
1         Cervecería                                     Manush Rosario   
2         Cervecería                               Mosto Somos Cervezas   
3         Cervecería                             Nº40 Mercado de Birras   
4         Cervecería                              Divague Small Brewery   
5         Cervecería                                        Birramp Bar   
6          Heladería                                     Rumo Heladería   
7          Heladería                        Gelatería Italiana San Remo   
8          Heladería                                Heladería "Catania"   
9          Heladería                                    Touche de Crème   
10         Heladería                                         Kapricho's   
11             Museo

## Formato necesario para el optimizador de rutas

In [9]:
import numpy as np

# 1. Extraer latitud y longitud de la columna 'coords'
df_final['lat'] = df_final['coords'].apply(lambda x: x['lat'] if isinstance(x, dict) else np.nan)
df_final['lon'] = df_final['coords'].apply(lambda x: x['lng'] if isinstance(x, dict) else np.nan)

# 2. Función para extraer apertura y cierre del Miércoles
def extraer_horario_miercoles(horarios, tipo_retorno):
    # Si es "No disponible" o no es un diccionario, devolvemos NA
    if not isinstance(horarios, dict) or horarios == "No disponible":
        return "NA"
    
    # Buscamos la entrada de 'Miércoles'
    miercoles = horarios.get('Miércoles', "Cerrado")
    
    # Si el lugar está cerrado ese día
    if miercoles == "Cerrado":
        return "Cerrado"
    
    # Si tiene datos (recordá que es una lista de turnos) tomamos el primero
    if isinstance(miercoles, list) and len(miercoles) > 0:
        if tipo_retorno == 'apertura':
            return miercoles[0]['apertura']
        else:
            return miercoles[0]['cierre']
            
    return "NA"

# 3. Aplicar la función para crear las nuevas columnas
df_final['Apertura'] = df_final['horarios_semana'].apply(lambda x: extraer_horario_miercoles(x, 'apertura'))
df_final['Cierre'] = df_final['horarios_semana'].apply(lambda x: extraer_horario_miercoles(x, 'cierre'))

# 4. Crear el ID (usando el índice) y seleccionar las columnas finales
df_final['ID'] = df_final.index
columnas_interes = ['ID', 'nombre', 'tipo', 'puntaje', 'lat', 'lon', 'Apertura', 'Cierre']
df_tabla_final = df_final[columnas_interes].copy()

# 5. Visualización
print(df_tabla_final.head())

   ID                  nombre              tipo  puntaje        lat  \
0   0        Rosario Santa Fe  Punto de Partida      3.9 -32.939191   
1   1          Manush Rosario        Cervecería      4.6 -32.932807   
2   2    Mosto Somos Cervezas        Cervecería      4.6 -32.946557   
3   3  Nº40 Mercado de Birras        Cervecería      4.6 -32.948995   
4   4   Divague Small Brewery        Cervecería      4.6 -32.936416   

         lon Apertura Cierre  
0 -60.672661       NA     NA  
1 -60.652391    17:30  01:30  
2 -60.658378    18:00  01:00  
3 -60.653310    18:00  00:30  
4 -60.654011    19:00  02:30  


In [29]:
# Guardamos la tabla final en un nuevo archivo .txt para análisis posterior
df_tabla_final.to_csv('data/g_nodos.txt', index=False, sep='\t')

## Cálculo matrices de distancia y tiempo

In [13]:
# Actualizamos datos lugares con los 21 lugares de interés (1 origen + 20 seleccionados)
datos_lugares = df_final.to_dict('records')

In [ ]:
import pandas as pd
import time

# 1. Preparar datos de los nodos
coords_solo = [d['coords'] for d in datos_lugares]
indices_solo = [d['ID'] for d in datos_lugares]
n = len(coords_solo)

# Matrices para almacenar valores puros
matriz_segundos = []
matriz_metros = []

print(f"Calculando matriz de {n}x{n} (Valores puros: Segundos y Metros)...")

# 2. Procesamiento por filas (Batching para evitar el límite de 100 elementos)
for i in range(n):
    origen_actual = [coords_solo[i]]
    
    try:
        # Eliminamos departure_time para que NO considere tráfico
        res_fila = gmaps.distance_matrix(
            origins=origen_actual,
            destinations=coords_solo,
            mode="driving",
            language="es"
        )
        
        segundos_fila = []
        metros_fila = []
        elementos = res_fila['rows'][0]['elements']
        
        for el in elementos:
            if el['status'] == 'OK':
                # Extraemos valores puros sin procesar
                val_segundos = el['duration']['value'] # Segundos
                val_metros = el['distance']['value']   # Metros
                
                segundos_fila.append(val_segundos)
                metros_fila.append(val_metros)
            else:
                segundos_fila.append(None)
                metros_fila.append(None)
        
        matriz_segundos.append(segundos_fila)
        matriz_metros.append(metros_fila)
        
        print(f"✔ Fila {i+1}/{n} completada.")
        
    except Exception as e:
        print(f"✘ Error en fila {i}: {e}")
        matriz_segundos.append([None] * n)
        matriz_metros.append([None] * n)

# 3. Crear DataFrames con los valores crudos
df_tiempo = pd.DataFrame(matriz_segundos, index=indices_solo, columns=indices_solo)
df_distancia = pd.DataFrame(matriz_metros, index=indices_solo, columns=indices_solo)

# 4. Resultados
print("\n--- MATRIZ DE DURACIÓN (SEGUNDOS - SIN TRÁFICO) ---")
print(df_tiempo)

print("\n--- MATRIZ DE DISTANCIA (METROS) ---")
print(df_distancia)

Calculando matriz de 21x21 (Valores puros: Segundos y Metros)...
✔ Fila 1/21 completada.
✔ Fila 2/21 completada.
✔ Fila 3/21 completada.
✔ Fila 4/21 completada.
✔ Fila 5/21 completada.
✔ Fila 6/21 completada.
✔ Fila 7/21 completada.
✔ Fila 8/21 completada.
✔ Fila 9/21 completada.
✔ Fila 10/21 completada.
✔ Fila 11/21 completada.
✔ Fila 12/21 completada.
✔ Fila 13/21 completada.
✔ Fila 14/21 completada.
✔ Fila 15/21 completada.
✔ Fila 16/21 completada.
✔ Fila 17/21 completada.
✔ Fila 18/21 completada.
✔ Fila 19/21 completada.
✔ Fila 20/21 completada.
✔ Fila 21/21 completada.

--- MATRIZ DE DURACIÓN (SEGUNDOS - SIN TRÁFICO) ---
      0    1    2    3    4     5     6     7     8    9   ...    11    12  \
0      0  563  592  640  444  1068   866   629   389  859  ...   274   827   
1    491    0  396  464  117   861   542   757   764  674  ...   723   724   
2    410  393    0  206  294   629   657   550   557  426  ...   517   487   
3    542  441  220    0  438   552   559   578   564  

In [ ]:
df_tiempo

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,0,563,592,640,444,1068,866,629,389,859,...,274,827,1115,1016,771,487,496,565,818,467
1,491,0,396,464,117,861,542,757,764,674,...,723,724,1000,824,590,148,653,1,510,110
2,410,393,0,206,294,629,657,550,557,426,...,517,487,800,579,331,301,863,394,509,303
3,542,441,220,0,438,552,559,578,564,246,...,516,286,599,387,249,360,919,442,427,447
4,495,119,377,332,0,755,461,762,769,545,...,710,607,920,696,458,121,687,120,423,120
5,840,592,744,643,709,0,246,1078,1111,609,...,1072,778,535,296,485,663,966,594,452,682
6,799,482,684,603,605,526,0,1072,1080,616,...,1032,776,666,484,451,518,856,484,257,614
7,494,707,405,572,602,995,981,0,447,493,...,508,437,607,829,691,636,947,709,869,625
8,455,704,482,648,585,1072,1030,355,0,680,...,75,638,890,906,768,628,543,705,946,607
9,671,594,343,282,591,618,653,469,533,0,...,579,156,378,470,336,513,1048,595,513,600


In [ ]:
df_distancia

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,0,3177,3076,3408,2635,5241,4548,3385,1713,4567,...,1358,4980,7219,5170,4062,2770,2481,3185,4207,2728
1,2749,0,1857,2245,511,4415,2888,4176,4446,3401,...,4089,3801,5920,3996,2900,646,3692,7,2456,627
2,2050,1900,0,909,1356,2742,3206,2907,3177,2063,...,2821,2467,4730,2650,1563,1489,4514,1907,2332,1448
3,3020,2344,991,0,2325,2404,2881,2756,2975,1161,...,2881,1560,3823,1760,1236,1934,5836,2351,2005,2417
4,2732,542,1838,1728,0,3565,2381,4154,4424,2886,...,4076,3290,5553,3481,2386,627,4404,549,2036,619
5,4292,3169,3464,3006,3680,0,948,5699,5986,3174,...,5632,4098,2924,1393,2158,3216,6483,3177,1762,3995
6,4009,2586,3116,2724,2826,2220,0,5428,5699,2840,...,5346,3768,3724,1837,1808,2438,5900,2594,935,2918
7,2756,4148,2244,3155,3605,4981,5457,0,2585,2730,...,3069,2836,4437,4337,3814,3738,5219,4156,4583,3729
8,2331,4407,2652,3563,3865,5389,5857,1865,0,3628,...,408,4043,6037,4745,4221,4000,3028,4414,4990,3957
9,3936,3247,1886,1481,3228,2793,3264,2524,3372,0,...,3780,919,2657,2152,1621,2837,6400,3254,2391,3321


In [ ]:
# Guardar df_distancias en un archivo CSV
df_distancia.to_csv('data/g_distancias.csv')

# Guardar df_tiempos en un archivo CSV
df_tiempo.to_csv('data/g_tiempos.csv')
